In [9]:
import os

folders = ['data', 'figures', 'notebooks', 'src']
for folder in folders:
    os.makedirs(folder, exist_ok=True)
    
print("Folder structure created!")

Folder structure created!


In [10]:
import GEOparse

# Download the dataset from NCBI GEO
gse = GEOparse.get_GEO(geo="GSE15605", destdir="./data")

print("Dataset downloaded successfully!")
print(f"Number of samples: {len(gse.gsms)}")
print(f"Number of platforms: {len(gse.gpls)}")

19-May-2026 21:52:01 DEBUG utils - Directory ./data already exists. Skipping.
19-May-2026 21:52:01 INFO GEOparse - File already exist: using local version.
19-May-2026 21:52:01 INFO GEOparse - Parsing ./data\GSE15605_family.soft.gz: 


19-May-2026 21:52:01 DEBUG GEOparse - DATABASE: GeoMiame
19-May-2026 21:52:01 DEBUG GEOparse - SERIES: GSE15605
19-May-2026 21:52:01 DEBUG GEOparse - PLATFORM: GPL570
c:\Users\25god\OneDrive - University of Pittsburgh\Projects\Melanoma Gene Expression\venv\Lib\site-packages\GEOparse\GEOparse.py:401: DtypeWarning: Columns (0: SPOT_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
19-May-2026 21:52:03 DEBUG GEOparse - SAMPLE: GSM390208
19-May-2026 21:52:03 DEBUG GEOparse - SAMPLE: GSM390209
19-May-2026 21:52:03 DEBUG GEOparse - SAMPLE: GSM390210
19-May-2026 21:52:03 DEBUG GEOparse - SAMPLE: GSM390211
19-May-2026 21:52:03 DEBUG GEOparse - SAMPLE: GSM390212
19-May-2026 21:52:03 DEBUG GEOparse - SAMPLE: GSM390213
19-May-2026 21:52:03 DEBUG GEOparse - SAMPLE: GSM390214
19-May-2026 21:52:03 DEBUG GEOparse - SAMPLE: GSM390215
19-May-2026 21:52:03 DEBUG GEOparse - SAMPLE: GSM390216
19-May-2026 21:52:03 DEBUG

Dataset downloaded successfully!
Number of samples: 74
Number of platforms: 1


In [11]:
#Looking at sameple names and their characteristics
for name, sample in list(gse.gsms.items())[:5]:
    print("Sample: " + name)
    print("Title: " + sample.metadata['title'][0])
    print("Source: " + sample.metadata['source_name_ch1'][0])
    print("-" * 40)

Sample: GSM390208
Title: Normal_skin N118
Source: Frozen tip of melanoma excision
----------------------------------------
Sample: GSM390209
Title: Normal_skin N157
Source: Frozen tip of melanoma excision
----------------------------------------
Sample: GSM390210
Title: Normal_skin N173
Source: Frozen tip of melanoma excision
----------------------------------------
Sample: GSM390211
Title: Normal_skin N201
Source: Frozen tip of melanoma excision
----------------------------------------
Sample: GSM390212
Title: Normal_skin N206
Source: Frozen tip of melanoma excision
----------------------------------------


In [12]:
import pandas as pd

#Exstract expression data from each sample into a dataframe
expression_data = pd.DataFrame()

for name, sample in gse.gsms.items():
    expression_data[name] = sample.table['VALUE']

#Use probe IDs as the index
expression_data.index = gse.gsms[list(gse.gsms.keys())[0]].table['ID_REF']

print(f"Expression matrix shape: rows={expression_data.shape[0]}, columns={expression_data.shape[1]}")
print("\nFirst few rows:")
expression_data.head()

Expression matrix shape: rows=44137, columns=74

First few rows:


,GSM390208,GSM390209,GSM390210,GSM390211,GSM390212,GSM390213,GSM390214,GSM390215,GSM390216,GSM390217,...,GSM390272,GSM390273,GSM390274,GSM390275,GSM390276,GSM390277,GSM390278,GSM390279,GSM390280,GSM390281
ID_REF,,,,,,,,,,,,,,,,,,,,,
1007_s_at,11.020532,10.950778,11.473811,10.822769,10.889048,11.102960,10.530630,10.541802,10.893512,11.323169,...,10.536372,9.789944,9.387112,10.581330,8.563208,10.054200,9.828538,9.284716,10.724639,10.302563
1053_at,8.031035,8.660447,8.150595,8.511551,8.680038,8.456048,7.861925,6.927419,8.906621,8.765491,...,8.659189,8.657827,9.327206,9.111500,8.561788,8.854245,8.546774,7.166762,9.624508,9.273964
117_at,6.973822,7.106163,8.156202,6.437214,6.372524,6.337900,7.736619,5.080306,6.917989,6.586833,...,8.960423,8.367059,7.786497,8.126415,7.488075,8.493820,7.410054,7.394527,7.087341,8.294484
121_at,6.175225,5.464620,6.433634,6.622263,6.456261,6.814377,6.368382,5.920322,7.059717,7.067290,...,6.433768,6.017126,5.751758,5.796932,6.841726,7.007100,5.966656,5.287752,5.781160,7.677899
1294_at,8.308560,8.395153,9.750258,8.246585,7.963901,8.057381,8.520238,7.913927,8.682314,9.178866,...,8.870373,9.070832,9.549103,8.806603,7.924857,8.303491,7.707022,9.365837,8.369271,7.051232


In [ ]:
#Create a dictionary to store sample labels
sample_labels = {}


#TODO: determine why there is an issue with assigning cell type

for name, sample in gse.gsms.items():
    characteristics = sample.metadata.get('characteristics_ch1', [])

    #Find the diagnosis field
    diagnosis = ''
    for char in characteristics:
        if 'diagnosis:' in char.lower():
            diagnosis = char.lower()
            break

    if 'normal' in diagnosis:
        sample_labels[name] = 'normal'
    elif 'metastatic' in diagnosis:
        sample_labels[name] = 'metastatic'
    else:
        sample_labels[name] = 'primary'

#Counting how many of each type are present:
from collections import Counter
print(Counter(sample_labels.values()))

Counter({'primary': 74})
